<a href="https://colab.research.google.com/github/namsarena/yelp-complaint-tagger/blob/main/02_relabel_distilabel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 · Install tools

In [ ]:
!pip install -q "distilabel[transformers]" datasets huggingface_hub

In [ ]:
# STEP 1 · Install tools (run once per session)
# distilabel = runs AI labelling in batches and keeps the results organised
# accelerate = helps load big models onto the GPU
# (Colab already includes transformers and torch, so we don't reinstall them)
!pip install -q distilabel accelerate

In [ ]:
# STEP 1b · Check versions and GPU (run after restarting)
import distilabel, transformers, torch
print("distilabel:", distilabel.__version__)
print("transformers:", transformers.__version__)
print("GPU ready:", torch.cuda.is_available())

# 02 · Relabel with a second AI (distilabel)
Goal: a second model (Phi-4-mini) labels the same 3,500 reviews that Qwen labelled.
Where both agree → keep. Where they disagree → a human checks.
Runtime: T4 GPU. Secrets needed: HF_TOKEN.

In [ ]:
# STEP 3 · Log in to Hugging Face and load the reviews Qwen already labelled
import pandas as pd
from google.colab import userdata                     # reads Colab 🔑 Secrets
from huggingface_hub import login, hf_hub_download, HfApi

In [ ]:
# 3a. Log in with the token stored in Secrets (never typed here)
login(userdata.get("HF_TOKEN"))

In [ ]:
# 3b. Where my data lives
REPO = "nrr26/yelp-complaint-labels"                  # my private dataset
api = HfApi()                                          # used later to save results

In [ ]:
# 3c. Download Qwen's labels file and open it as a table
path = hf_hub_download(REPO, "llm_labels_v2.csv", repo_type="dataset")
df = pd.read_csv(path)

In [ ]:
# 3d. Show what we got
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

## Step 4 · Rules, prompt and pipeline
Keep only restaurant reviews (~2,361). Write the label rules and the question we ask the AI.
Build the pipeline: Phi-4-mini on the GPU, greedy (same answer every run), 50 reviews per batch.
Nothing is labelled yet; the model loads when we run it in Step 5.

In [ ]:
# STEP 4 · Rules, prompt and pipeline (run once per session)
from datasets import Dataset
from distilabel.models import TransformersLLM
from distilabel.pipeline import Pipeline
from distilabel.steps.tasks import TextGeneration

In [ ]:
# 4a. Keep only restaurant reviews (the ones the model trains on)
rest = df[df["is_restaurant"].astype(str).str.lower().isin(["true", "1", "yes"])].reset_index(drop=True)
print("Restaurant reviews to label:", len(rest))

In [ ]:
# 4b. The 8 labels with short definitions (same as labels.yaml v1.1)
LABELS = {
    "wait_time": "slow seating, food or bill; forgotten or ignored by staff",
    "staff_attitude": "rude, dismissive or disrespectful staff (not slowness, not mistakes)",
    "food_quality": "taste, freshness, dryness, cooking, temperature, drinks made badly",
    "order_accuracy": "wrong, missing or sold-out items",
    "price_value": "overpriced, tiny portions, not worth it",
    "hygiene_safety": "dirty, hair or bugs in food, got sick",
    "ambience": "noise, music, seating, temperature, decor",
    "billing_policy": "wrong bill, refunds, reservation rules, manager handling",
}
RULES = """- Tag complaints only. Praise or neutral description gets no label.
- Slow service is wait_time, not staff_attitude.
- Wrong, missing or sold-out items are order_accuracy, not food_quality.
- Tag what is stated, not what is guessed. Use [] if there is no complaint."""

In [ ]:
# STEP 4b · Prompt v2: softer "empty" rule + 5 invented examples (replaces make_prompt)
EXAMPLES = [
    ("Our pasta was cold and the server ignored us for 20 minutes.", ["food_quality", "wait_time"]),
    ("Bland, overpriced tacos. Never again.", ["food_quality", "price_value"]),
    ("They forgot my fries and the manager refused to refund me.", ["order_accuracy", "billing_policy"]),
    ("Cute decor and friendly staff, but the music was so loud we couldn't talk.", ["ambience"]),
    ("Meh. Nothing special, just mediocre food.", ["food_quality"]),
]

RULES_V2 = """- These are 1-2 star reviews, so almost every one has at least one complaint.
  Use [] only if the review truly complains about nothing.
- Vague words count: "sub-par", "mediocre", "not appealing", "awful" about food = food_quality.
- Tag complaints only. Praise ("cute decor", "nice waitress") gets no label.
- Slow service is wait_time, not staff_attitude.
- Wrong, missing or sold-out items are order_accuracy, not food_quality."""

def make_prompt(review):
    label_lines = "\n".join(f"- {k}: {v}" for k, v in LABELS.items())
    shots = "\n".join(f"Review: {t}\nAnswer: {json.dumps(l)}" for t, l in EXAMPLES)
    return (f"Tag this restaurant review with complaint labels.\n\nLabels:\n{label_lines}\n\n"
            f"Rules:\n{RULES_V2}\n\nExamples:\n{shots}\n\n"
            f"Review: {review}\nAnswer:")

print(make_prompt("The soup was salty and we waited forever.")[-400:])   # peek at the end of the new prompt
print("\n✅ prompt v2 ready")

In [ ]:
# 4c. The question we ask the AI for each review
def make_prompt(review):
    label_lines = "\n".join(f"- {k}: {v}" for k, v in LABELS.items())
    return (f"Tag this restaurant review with complaint labels.\n\nLabels:\n{label_lines}\n\n"
            f"Rules:\n{RULES}\n\nReview: {review}\n\n"
            'Answer with a JSON list of labels only, for example ["wait_time", "food_quality"].')

In [ ]:
# 4d. Build the pipeline: which model, how it answers, batch size
with Pipeline(name="second-labeller") as pipeline:
    TextGeneration(
        llm=TransformersLLM(
            model="microsoft/Phi-4-mini-instruct",   # the second AI labeller (~3.8B parameters)
            device="cuda",                           # run on the T4 GPU
            torch_dtype="float16",                   # the number format T4 GPUs handle well
            generation_kwargs={"max_new_tokens": 60, "do_sample": False},  # short answer, same every run
        ),
        input_batch_size=50,                         # reviews per batch
    )

In [ ]:
# 4e. Show one example prompt so we can read what the AI will see
print("\n--- Example prompt ---\n", make_prompt(rest.loc[0, "text"])[:900])
print("\n✅ pipeline ready")

## Step 5 · Test run (20 reviews)
First run downloads Phi-4-mini (~8 GB, a few minutes). Then it labels 20 reviews.
Compare Phi's labels with Qwen's, and time it to estimate the full run.


In [ ]:
# STEP 5 · Load Phi-4-mini and define the labelling function (run once per session)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "microsoft/Phi-4-mini-instruct"

In [ ]:
# 5a. The tokenizer turns text into numbers the model reads (and back again)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"                 # pad on the left so every answer starts at the same place
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [ ]:
# 5b. Load the model onto the GPU in float16 (the format T4 GPUs handle well)
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda").eval()

In [ ]:
# 5c. Label a small batch of reviews in one go
def label_batch(texts):
    chats = [[{"role": "user", "content": make_prompt(t)}] for t in texts]   # one chat message per review
    enc = tok.apply_chat_template(chats, add_generation_prompt=True, padding=True,
                                  return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():                                                  # labelling only, no learning
        out = llm.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.pad_token_id)
    start = enc["input_ids"].shape[1]                                      # skip the prompt, keep the answer
    return [tok.decode(o[start:], skip_special_tokens=True).strip() for o in out]

print("✅ Phi-4-mini loaded. GPU memory used:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

In [ ]:
# STEP 6 · Test on 20 reviews and compare with Qwen
import time, json, ast

sample = rest.head(20)
BATCH = 10                                     # reviews sent to the GPU at once

In [ ]:
# Check all currently defined variables in the kernel
%whos

In [ ]:
# 6b. Turn answer text into a real list (empty set if it isn't clean JSON)
def to_set(text):
    try:
        return set(json.loads(text))
    except Exception:
        return None                            # None = answer wasn't a clean list


In [ ]:
# 6c. Side-by-side comparison
same = 0
for (_, row), phi in zip(sample.iterrows(), answers):
    qwen = set(ast.literal_eval(row["labels"])) if isinstance(row["labels"], str) else set()
    p = to_set(phi)
    ok = p is not None and p == qwen
    same += ok
    print(f"{'✅' if ok else '❌'} Qwen: {str(sorted(qwen)):<40} Phi: {phi[:40]:<40} | {row['text'][:55]}")
print(f"\nAgreement: {same}/20 exactly the same")

In [ ]:
# GPU memory check: the model alone should use about 7–8 GB
import torch
used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU memory used: {used:.1f} GB of {total:.1f} GB")

In [ ]:
# STEP 6 · Test on 20 reviews and compare Phi with Qwen (ONE cell — run it whole)
import time, json, re

sample = rest.head(20)
BATCH = 4                                        # reviews sent to the GPU at once

# 6a. Label 20 reviews with Phi, 10 at a time, and time it
t0 = time.time()
answers = []
for s in range(0, len(sample), BATCH):
    answers += label_batch(sample["text"].iloc[s:s + BATCH].tolist())
secs = time.time() - t0
print(f"⏱ {secs:.0f} s for 20 reviews → full run of {len(rest):,} ≈ {secs / 20 * len(rest) / 60:.0f} min\n")

# so. Read labels from any format: "a, b" text, a JSON list, or empty (NaN)
def read_labels(text):
    if not isinstance(text, str):
        return set()                              # NaN / empty = no complaint labels
    return set(re.findall(r"[a-z_]+", text)) & set(LABELS)   # keep only the 8 known label names

# 6c. Side-by-side comparison
same = 0
for (_, row), phi in zip(sample.iterrows(), answers):
    qwen, p = read_labels(row["labels"]), read_labels(phi)
    ok = qwen == p
    same += ok
    print(f"{'✅' if ok else '❌'} Qwen: {', '.join(sorted(qwen)) or '—':<45} Phi: {', '.join(sorted(p)) or '—':<45} | {row['text'][:50]}")
print(f"\nAgreement: {same}/20 exactly the same")

In [ ]:
# STEP 6b · Show Phi's raw answer where we read no labels, and a softer agreement score
import numpy as np

# 1. Raw text Phi wrote when we found no labels in it
for i, ((_, row), phi) in enumerate(zip(sample.iterrows(), answers)):
    if not read_labels(phi):
        print(f"#{i}  PHI RAW: {phi!r}")
        print(f"     REVIEW: {row['text'][:140]}\n")

# 2. Softer score: overlap between the two label sets (1.0 = identical, 0 = nothing shared)
overlap = []
for (_, row), phi in zip(sample.iterrows(), answers):
    q, p = read_labels(row["labels"]), read_labels(phi)
    overlap.append(len(q & p) / max(1, len(q | p)))
print("Average overlap:", round(float(np.mean(overlap)), 2))

## Step 7 · Full run with Phi-4-mini, prompt v2
Test result: overlap 0.77, 1/20 empty. Labels all 2,361 restaurant reviews in chunks of 500.
Each chunk is saved to my private dataset (phi_labels_v2/) as it finishes.
If Colab disconnects: re-run Steps 3 → 4 → 4b → 5 → 6, then this cell. Saved chunks are skipped.

In [ ]:
# STEP 7 · Full run: label all restaurant reviews with Phi (prompt v2), saved chunk by chunk
import time, re
import pandas as pd

# Helper: read labels from any answer format (same as Step 6)
def read_labels(text):
    if not isinstance(text, str):
        return set()
    return set(re.findall(r"[a-z_]+", text)) & set(LABELS)

CHUNK, BATCH = 500, 4                 # 500 reviews per saved file; 4 at a time on the GPU
FOLDER = "phi_labels_v2"              # folder inside my private dataset

# Which chunks are already saved? (so a re-run skips them)
saved = {f for f in api.list_repo_files(REPO, repo_type="dataset") if f.startswith(FOLDER + "/")}

for start in range(0, len(rest), CHUNK):
    name = f"{FOLDER}/chunk_{start:05d}.csv"
    if name in saved:
        print("⏭ skip (already saved)", name)
        continue

    # Label this chunk, 4 reviews at a time
    part = rest.iloc[start:start + CHUNK]
    t0, raw = time.time(), []
    for s in range(0, len(part), BATCH):
        raw += label_batch(part["text"].iloc[s:s + BATCH].tolist())

    # Keep: review id, Phi's raw answer, cleaned labels
    out = pd.DataFrame({"id": part["id"].values,
                        "phi_raw": raw,
                        "phi_labels": [", ".join(sorted(read_labels(r))) for r in raw]})

    # Save this chunk to my private Hugging Face dataset
    out.to_csv("chunk.csv", index=False)
    api.upload_file(path_or_fileobj="chunk.csv", path_in_repo=name, repo_id=REPO,
                    repo_type="dataset", commit_message=f"Phi-4-mini labels (prompt v2) {name}")
    print(f"✅ saved {name}: {len(part)} reviews in {time.time() - t0:.0f}s")

print("🎉 all chunks done")

## Step 8 · Compare Phi vs Qwen and make the human-review sheet
Merge both AIs' labels by id → agreement per label → sort reviews into Agree / Close / Disagree
→ save merged table to my private dataset → download human_review.csv for Google Sheets.

In [ ]:
# STEP 8 · Compare the two AI labellers and export a human-review sheet
import pandas as pd, re
from huggingface_hub import hf_hub_download
from sklearn.metrics import cohen_kappa_score
from google.colab import files

def read_labels(text):
    if not isinstance(text, str):
        return set()
    return set(re.findall(r"[a-z_]+", text)) & set(LABELS)

# 8a. Load Phi's 5 saved chunks from my dataset and stack them into one table
names = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.startswith("phi_labels_v2/"))
phi = pd.concat([pd.read_csv(hf_hub_download(REPO, n, repo_type="dataset")) for n in names])
print("Phi rows loaded:", len(phi))

# 8b. Join Phi's labels to the reviews and Qwen's labels, matching on review id
m = rest[["id", "text", "labels"]].merge(phi[["id", "phi_labels"]], on="id")
m["qwen"] = m["labels"].apply(read_labels)
m["phi"] = m["phi_labels"].apply(read_labels)

# 8c. Agreement per label: % same, and Cohen's kappa (agreement beyond chance)
print(f"\n{'label':<16}{'Qwen':>6}{'Phi':>6}{'same %':>9}{'kappa':>8}")
for lab in LABELS:
    q = m["qwen"].apply(lambda s: lab in s).astype(int)
    p = m["phi"].apply(lambda s: lab in s).astype(int)
    print(f"{lab:<16}{q.sum():>6}{p.sum():>6}{(q == p).mean() * 100:>8.0f}%{cohen_kappa_score(q, p):>8.2f}")

# 8d. Sort every review into a group by how many labels differ
m["n_diff"] = [len(q ^ p) for q, p in zip(m["qwen"], m["phi"])]          # ^ = labels only one AI chose
m["group"] = m["n_diff"].map(lambda d: "agree" if d == 0 else "close" if d == 1 else "disagree")
print("\nGroups:\n", m["group"].value_counts().to_string())

# 8e. Save the full merged table to my private dataset (ids + labels only)
m["qwen_labels"] = m["qwen"].apply(lambda s: ", ".join(sorted(s)))
m["phi_labels"] = m["phi"].apply(lambda s: ", ".join(sorted(s)))
m[["id", "qwen_labels", "phi_labels", "n_diff", "group"]].to_csv("merged_qwen_phi.csv", index=False)
api.upload_file(path_or_fileobj="merged_qwen_phi.csv", path_in_repo="merged_qwen_phi_v1.csv",
                repo_id=REPO, repo_type="dataset", commit_message="Merge Qwen + Phi labels, agreement groups")

# 8f. Human-review sheet: ALL "disagree" rows + 50 random "close" rows to spot-check
review = pd.concat([m[m["group"] == "disagree"],
                    m[m["group"] == "close"].sample(min(50, (m["group"] == "close").sum()), random_state=42)])
sheet = review[["id", "group", "qwen_labels", "phi_labels", "text"]].copy()
sheet.insert(4, "final_labels", "")          # you type the correct labels here
sheet.insert(5, "notes", "")                 # why, or what was confusing
sheet.to_csv("human_review.csv", index=False)
print(f"\nHuman-review sheet: {len(sheet)} rows → downloading human_review.csv")
files.download("human_review.csv")

In [ ]:
# Step 9: load my human labels from the private HF dataset
import pandas as pd
from huggingface_hub import hf_hub_download

# Ensure REPO is defined if the session restarted
REPO = "nrr26/yelp-complaint-labels"

human = pd.read_csv(hf_hub_download(REPO, "human_labels_v1.csv", repo_type="dataset"))
human = human.fillna("")
print(len(human), "labelled reviews")
print(human["closer_to"].value_counts())
human.head()

In [ ]:
# Step 10: compare Qwen and Phi against my human labels
from sklearn.metrics import f1_score
import pandas as pd

# Fallback definition of LABELS if the cell session restarted
try:
    _ = LABELS
except NameError:
    LABELS = {
        "wait_time": "slow seating, food or bill; forgotten or ignored by staff",
        "staff_attitude": "rude, dismissive or disrespectful staff (not slowness, not mistakes)",
        "food_quality": "taste, freshness, dryness, cooking, temperature, drinks made badly",
        "order_accuracy": "wrong, missing or sold-out items",
        "price_value": "overpriced, tiny portions, not worth it",
        "hygiene_safety": "dirty, hair or bugs in food, got sick",
        "ambience": "noise, music, seating, temperature, decor",
        "billing_policy": "wrong bill, refunds, reservation rules, manager handling",
    }

def to_set(x):
    return set(l for l in str(x).split(", ") if l)

def jaccard(a, b):
    return 1.0 if not a and not b else len(a & b) / len(a | b)

H = human["human_labels"].map(to_set)
rows = []
for model in ["qwen_labels", "phi_labels"]:
    M = human[model].map(to_set)
    rows.append({
        "model": model.replace("_labels", ""),
        "exact": sum(h == m for h, m in zip(H, M)),
        "one_match": sum(bool(h & m) or (not h and not m) for h, m in zip(H, M)),
        "jaccard_avg": round(sum(jaccard(h, m) for h, m in zip(H, M)) / len(H), 2),
        "per_label_%": round(100 * sum(8 - len(h ^ m) for h, m in zip(H, M)) / (8 * len(H)), 1),
    })
print(pd.DataFrame(rows).to_string(index=False), "\n")

# Per-label F1: which labels each model gets wrong
f1 = {}
for model in ["qwen_labels", "phi_labels"]:
    M = human[model].map(to_set)
    f1[model.replace("_labels", "")] = {
        l: round(f1_score([l in h for h in H], [l in m for m in M], zero_division=0), 2)
        for l in LABELS}
print(pd.DataFrame(f1))

In [ ]:
# Step 11: setup + rules v1.2 + prompt v3 + load Qwen
import torch, re, gc, pandas as pd
from google.colab import userdata
from huggingface_hub import login, HfApi, hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import f1_score

login(userdata.get("HF_TOKEN"))
REPO, CH, BATCH = "nrr26/yelp-complaint-labels", 500, 4
api = HfApi()
df = pd.read_csv(hf_hub_download(REPO, "llm_labels_v2.csv", repo_type="dataset"))
rest = df[df["is_restaurant"].astype(str).str.lower().isin(["true", "1", "yes"])].reset_index(drop=True)
human = pd.read_csv(hf_hub_download(REPO, "human_labels_v1.csv", repo_type="dataset")).fillna("")
test = human.merge(rest[["id", "text"]], on="id")
LABELS = ["wait_time", "staff_attitude", "food_quality", "order_accuracy",
          "price_value", "hygiene_safety", "ambience", "billing_policy"]

PROMPT_V3 = """You tag complaints in a 1-2 star restaurant review. Each label has a test question:
- wait_time: Was something slow (seating, food, bill), or were they ignored or forgotten?
- staff_attitude: Was a staff member rude, dismissive or hostile? Slow, forgetful or making mistakes is NOT rude.
- food_quality: Was the food or drink bad: taste, freshness, cooking, temperature?
- order_accuracy: Did they receive something different from what they ordered, or was an item missing or sold out? Bad, cold or small food is NOT order_accuracy.
- price_value: Did it cost too much for what they got? "Overpriced", "not worth it", "for that price", small portions. Count it even as a side remark.
- hygiene_safety: Dirty place or dishes, hair or bugs in food, or got sick?
- ambience: Would fixing the room fix it: noise, music, seating, room temperature, decor, smell? Praise of the room is NOT a complaint. Dirty is hygiene, not ambience.
- billing_policy: Wrong bill, refund, reservation or cancellation rule, or a manager handling a dispute badly?

Rules:
1. Tag only complaints the reviewer actually makes. Praise is never a complaint.
2. Tag every complaint present, including side remarks.
3. These are 1-2 star reviews, so most have at least one complaint. Answer none only if no label fits.

Examples:
Review: "Gorgeous decor and great music, but the steak was raw and the waiter never came back."
Answer: food_quality, wait_time
Review: "I ordered the veggie burger and got a beef one. $18 for that? No thanks."
Answer: order_accuracy, price_value
Review: "Our server was slow and kept forgetting things, but she was sweet."
Answer: wait_time
Review: "The hostess rolled her eyes and told us to eat somewhere else."
Answer: staff_attitude
Review: "Tiny portions, and they charged us for drinks we never ordered."
Answer: price_value, billing_policy
Review: "It was so loud we couldn't talk and the tables were sticky."
Answer: ambience, hygiene_safety

Now tag this review. Reply only with label names separated by commas, or none.
Review: "{text}"
Answer:"""

def load_model(name):
    global tok, llm
    if "llm" in globals():
        del globals()["llm"]; gc.collect(); torch.cuda.empty_cache()
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="cuda").eval()

def label_batch(texts):
    chats = [[{"role": "user", "content": PROMPT_V3.replace("{text}", t)}] for t in texts]
    enc = tok.apply_chat_template(chats, add_generation_prompt=True, padding=True,
                                  return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=60, do_sample=False,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def label_all(texts):
    raw = []
    for i in range(0, len(texts), BATCH):
        raw += label_batch(list(texts[i:i + BATCH]))
    return raw

def run_chunks(folder, col):
    """Label every restaurant review in chunks of 500, saved to HF; resumes after a crash."""
    for start in range(0, len(rest), CH):
        name = f"{folder}/chunk_{start:05d}.csv"
        if api.file_exists(REPO, name, repo_type="dataset"):
            print("⏭️ skip", name); continue
        part = rest.iloc[start:start + CH]
        out = pd.DataFrame({"id": part["id"].values, col: label_all(part["text"].tolist())})
        api.upload_file(path_or_fileobj=out.to_csv(index=False).encode(), path_in_repo=name,
                        repo_id=REPO, repo_type="dataset")
        print("✅ saved", name)
    files = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.startswith(folder + "/"))
    return pd.concat([pd.read_csv(hf_hub_download(REPO, f, repo_type="dataset")) for f in files])

read_labels = lambda s: set(re.findall(r"[a-z_]+", str(s))) & set(LABELS)
to_set = lambda x: set(l for l in str(x).split(", ") if l)
jac = lambda a, b: 1.0 if not a and not b else len(a & b) / len(a | b)
H = test["human_labels"].map(to_set)

load_model("Qwen/Qwen3-4B-Instruct-2507")
print(len(df), "rows |", len(rest), "restaurant |", len(test), "human labels | Qwen loaded ✅")

In [ ]:
# Step 11b: prompt v4 (strict). Overwrites the prompt that label_batch uses.
PROMPT_V3 = """You tag complaints in a 1-2 star restaurant review. Be strict: tag only SPECIFIC, CONCRETE complaints.
Most reviews have 1 or 2 labels. Vague remarks get no label: "service was awful", "food was meh", "not great", "won't be back".

Labels and their test question:
- wait_time: Does the reviewer describe a delay (seating, food, bill) or being ignored? e.g. "waited 40 minutes", "nobody came to our table"
- staff_attitude: Does the reviewer describe rude words or behaviour by a person? e.g. "rolled her eyes", "yelled at us". Slow, forgetful, mistakes or "bad service" alone are NOT staff_attitude.
- food_quality: Does the reviewer name a specific food or drink problem? e.g. stale, greasy, bland, burnt, cold, raw. "Meh" or "not great" alone is NOT enough.
- order_accuracy: Did they get something different from what they ordered, or was an item missing or sold out? Bad food is NOT order_accuracy.
- price_value: Does the reviewer say it costs too much for what they got? e.g. "overpriced", "not worth $20", "tiny portion for the price".
- hygiene_safety: Dirty place or dishes, hair or bugs in food, or got sick?
- ambience: Would fixing the room fix it: noise, music, seating comfort, room temperature, smell? Busy or packed is NOT ambience. Praise is never a complaint.
- billing_policy: Wrong bill, refund, a reservation or cancellation rule, or a manager handling a dispute badly?

Examples:
Review: "Gorgeous decor and great music, but the steak was raw and the waiter never came back."
Answer: food_quality, wait_time
Review: "Service was awful and the food was meh. Way overpriced for what you get."
Answer: price_value
Review: "Packed with tourists. Greasy noodles, and it took 15 minutes before anyone took our order."
Answer: food_quality, wait_time
Review: "The hostess rolled her eyes and told us to eat somewhere else."
Answer: staff_attitude
Review: "Third time they were out of the fish tacos, and they charged us for drinks we never ordered."
Answer: order_accuracy, billing_policy
Review: "Used to love this place. Smaller menu now and everything is just okay."
Answer: none

Now tag this review. Reply only with label names separated by commas, or none.
Review: "{text}"
Answer:"""
print("prompt v4 set ✅")

In [ ]:
# Step 12: Qwen v2 vs v3 against my human labels
runs = {"qwen_v2": test["qwen_labels"].map(to_set),
        "qwen_v3": pd.Series([read_labels(r) for r in label_all(test["text"].tolist())])}
for name, M in runs.items():
    print(f"{name}: jaccard {sum(jac(h, m) for h, m in zip(H, M)) / len(H):.2f} | exact {sum(h == m for h, m in zip(H, M))}/{len(H)}")
print(pd.DataFrame({name: {l: round(f1_score([l in h for h in H], [l in m for m in M], zero_division=0), 2)
                           for l in LABELS} for name, M in runs.items()}))

In [ ]:
# Step 12b: diagnose v3 (reuses Step 12 results, no GPU)
V2, V3 = runs["qwen_v2"], runs["qwen_v3"]
avg = lambda S: round(sum(len(s) for s in S) / len(S), 2)
print("avg labels per review → human", avg(H), "| v2", avg(V2), "| v3", avg(V3), "\n")

rows = []
for l in LABELS:
    for name, M in runs.items():
        rows.append({"label": l, "run": name,
                     "extra": sum(l in m and l not in h for h, m in zip(H, M)),
                     "missed": sum(l in h and l not in m for h, m in zip(H, M))})
print(pd.DataFrame(rows).pivot(index="label", columns="run", values=["extra", "missed"]))

test["v2"], test["v3"], test["H"] = V2.values, V3.values, H.values
test["loss"] = [jac(h, a) - jac(h, b) for h, a, b in zip(H, V2, V3)]
for _, r in test.sort_values("loss", ascending=False).head(5).iterrows():
    print(f"\n#{r.id}  human={sorted(r.H)}  v2={sorted(r.v2)}  v3={sorted(r.v3)}\n{r.text[:300]}…")

In [ ]:
# Step 12c: score prompt v4 next to v2 and v3
runs["qwen_v4"] = pd.Series([read_labels(r) for r in label_all(test["text"].tolist())])
for name, M in runs.items():
    print(f"{name}: jaccard {sum(jac(h, m) for h, m in zip(H, M)) / len(H):.2f} | exact {sum(h == m for h, m in zip(H, M))}/{len(H)} | avg labels {avg(M)}")
print(f"human avg labels {avg(H)}\n")
print(pd.DataFrame({name: {l: round(f1_score([l in h for h in H], [l in m for m in M], zero_division=0), 2)
                           for l in LABELS} for name, M in runs.items()}))
